# Adaptation Curve Fitting

Supported cells start from the curated time-series data under `data/time-series/` and write generated files under ignored `outputs/` directories.

Upstream image/video processing, raw acquisition data, segmentation outputs, bead-center detection outputs, and other large intermediates are intentionally not included in this repository.

Scientific goal: quantify how bead-tethered cell rotation changes after a sustained sucrose osmotic upshift. Rotation speed is used as the motor-output readout, normalized to the pre-shock baseline, and fit with simple exponential/sigmoidal summaries of collapse and recovery. These fitted parameters summarize adaptation dynamics; they are not intended as mechanistic motor models.

The committed bead traces are `0.1 s` mean-binned. The helper functions below materialize a temporary legacy-shaped CSV tree under `outputs/` so the original notebook logic can run without restoring the large upstream bead-processing outputs.


## Adaptation Assay

The manifest keeps the historical assay label `Adaption`; this notebook refers to the same sucrose adaptation experiment.

## Population Mean Traces Across Sucrose Concentrations

This section pools single-cell traces by sucrose concentration, normalizes each trace to its pre-shock speed, and plots the mean response. The shaded region marks the sustained osmotic challenge beginning at `175 s`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import median_filter
from scipy.optimize import curve_fit
import sys

# Allow the notebook to run both from this folder and from the repository root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "bead_time_series.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
else:
    sys.path.insert(0, str((NOTEBOOK_DIR / "code" / "bead-assay").resolve()))
from bead_time_series import ROOT, legacy_speed_tree, legacy_condition_folder, read_legacy_speed_csv, iter_condition_traces, median_kernel_size



In [ ]:

# --- Exponential decrease model ---
# Used to summarize the rapid motor-speed collapse after osmotic upshift.
# A is the drop amplitude, tau is the collapse timescale, C is the lower
# asymptote, and t0 allows a small horizontal offset around shock onset.
def exp_decrease(t, A, tau, C, t0):
    return A / (1 + np.exp((t - 175 - t0) / tau)) + C

# --- Paths ---
# Build a temporary legacy-style speed tree from compact Parquet inputs.
# The original source data used the historical spelling "Adaption".
parent_dir = legacy_speed_tree(assay="Adaption")

# --- Storage ---
all_data = []
all_times = []
concentrations = []

# Keep the original 300 Hz frame reference so legacy frame/300 code maps to
# seconds, even though the committed traces are already 0.1 s mean-binned.
sampling_rate = 300  # Hz frame reference

# --- Process each concentration folder ---
for subfolder in sorted(os.listdir(parent_dir)):
    print(f"\nProcessing: {subfolder}")
    subfolder_path = parent_dir / subfolder

    if not subfolder_path.is_dir():
        print(f"  Skipping {subfolder}: not a directory")
        continue

    csv_files = [f for f in os.listdir(subfolder_path) if f.endswith('.csv')]
    if not csv_files:
        print(f"  No CSV files found in {subfolder_path}")
        continue

    print(f"  Found {len(csv_files)} CSV files.")
    all_traces = []
    trace_times = []

    for csv_file in csv_files:
        file_path = subfolder_path / csv_file
        df = read_legacy_speed_csv(file_path)  # no headers

        if df.shape[1] < 2:
            print(f"    Skipping {csv_file}: Less than 2 columns")
            continue

        frame = df.iloc[:, 0].values
        speed = df.iloc[:, 1].values

        if len(speed) != len(frame):
            print(f"    Skipping {csv_file}: Time/speed length mismatch")
            continue

        # Convert the preserved frame reference to time in seconds.
        time = frame / sampling_rate

        # Apply approximately 1 s smoothing, matching the original notebook
        # intent while adapting the kernel size to the binned time base.
        kernel_size = median_kernel_size(time, window_s=1.0)
        speed = median_filter(speed, size=kernel_size, mode='nearest')

        # Normalize by the pre-shock baseline. Values after this step are
        # relative motor speeds, where 1.0 is the average before shock.
        baseline_region = speed[time < 180]
        baseline = np.nanmean(baseline_region)
        if np.isnan(baseline) or baseline == 0:
            print(f"    Skipping {csv_file}: Invalid baseline")
            continue
        speed_norm = speed / baseline

        all_traces.append(speed_norm)
        trace_times.append(time)

    if not all_traces:
        print(f"  No valid traces found in {subfolder}")
        continue

    print(f"  Processed {len(all_traces)} valid traces.")

    # --- Trim all traces to shortest length ---
    min_len = min(len(tr) for tr in all_traces)
    all_traces_trimmed = np.array([tr[:min_len] for tr in all_traces])
    time_trimmed = trace_times[0][:min_len]  # assume same start time

    # Average and std
    mean_speed = np.nanmean(all_traces_trimmed, axis=0)
    std_speed = np.nanstd(all_traces_trimmed, axis=0)

    all_times.append(time_trimmed)
    all_data.append(mean_speed)
    try:
        conc_val = int(subfolder.replace("mM", "").strip())
    except ValueError:
        conc_val = subfolder
    concentrations.append(conc_val)

    # --- Fit collapse after shock onset ---
    # The 175-240 s window captures the fast drop immediately after sucrose
    # addition without including the later recovery/adaptation phase.
    mask_decrease = (time_trimmed >= 175) & (time_trimmed <= 240)
    try:
        popt, _ = curve_fit(
            exp_decrease,
            time_trimmed[mask_decrease],
            mean_speed[mask_decrease],
            p0=[1 - np.min(mean_speed[mask_decrease]), 10, np.min(mean_speed[mask_decrease]), 10]
        )
        fit_curve = exp_decrease(time_trimmed, *popt)
        print(f"    Fit successful: tau = {popt[1]:.2f}, t0 = {popt[3]:.2f}")
    except Exception as e:
        print(f"    Fit failed for {subfolder}: {e}")
        fit_curve = None

# --- Plot averaged traces ---
plt.rcParams.update({'font.size': 25, 'font.family': 'Arial'})
fig, ax = plt.subplots(figsize=(10, 5.5))

for axis in ['top', 'bottom', 'left', 'right']:
    ax.spines[axis].set_linewidth(2)
ax.tick_params(axis='x', direction='in', width=2, length=7.5, pad=8)
ax.tick_params(axis='y', direction='in', width=2, length=7.5, pad=8)

for time_arr, mean_arr, conc in zip(all_times, all_data, concentrations):
    ax.plot(time_arr, mean_arr, label=f'{conc} mM')

# Stimulus shading: osmotic upshift begins at 175 s and remains on.
ax.axvspan(175, 1100, color="lightgray", alpha=0.6)
ax.axvline(175, linestyle='--', color='gray')
# ax.axvline(270, linestyle='--', color='gray')

ax.set_xlabel('Time (s)', fontsize=25)
ax.set_ylabel('Rotation Speed (Normalized)', fontsize=25)
ax.set_xlim(0, 1100)
ax.set_ylim(0, 1.15)
ax.legend(
    loc='lower center',
    bbox_to_anchor=(0.5, 1.02),   # center above plot
    ncol=2,                       # break into 2 columns
    frameon=False,
    fontsize=22,
    columnspacing=1.0,
    handlelength=2.0,
    handletextpad=0.5
)
ax.set_title("Adaptation to sucrose osmotic shock")

# Save plot
output_dir = ROOT / "outputs" / "bead" / "adaption"
os.makedirs(output_dir, exist_ok=True)
plt.savefig(output_dir / "osmopro_adaptation_all.pdf", dpi=350, bbox_inches='tight')
plt.show()

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from scipy.optimize import curve_fit

# --- Define exponential increase model ---
def exp_increase(t, B, tau, D):
    return B * (1 - np.exp(-(t - 200) / tau)) + D

# --- Sampling rate ---
sampling_rate = 300  # Hz

# --- Parent directory with condition subfolders ---
parent_dir = legacy_speed_tree(assay="Adaption")
print("Resolved parent_dir:", parent_dir.resolve())

# --- Output directory ---
output_dir = ROOT / "outputs" / "bead" / "adaption"
os.makedirs(output_dir, exist_ok=True)

# --- Storage for results ---
results = []

# --- Loop through each condition folder ---
for subfolder in sorted(os.listdir(parent_dir)):
    subfolder_path = parent_dir / subfolder
    print(f"\nProcessing: {subfolder}")

    if not subfolder_path.is_dir():
        print(f"  Skipping {subfolder}: not a directory.")
        continue

    try:
        concentration = int(subfolder.replace("mM", ""))
    except ValueError:
        print(f"  Skipping {subfolder}: not a valid concentration folder.")
        continue

    csv_files = [f for f in os.listdir(subfolder_path) if f.endswith(".csv")]
    if not csv_files:
        print(f"  No CSV files found in {subfolder_path}")
        continue

    for csv_file in csv_files:
        file_path = subfolder_path / csv_file
        try:
            df = read_legacy_speed_csv(file_path)
            df = df.apply(pd.to_numeric, errors='coerce').dropna()
        except Exception as e:
            print(f"    Skipping {csv_file}: failed to read CSV ({e})")
            continue

        if df.shape[1] < 2:
            print(f"    Skipping {csv_file}: Less than 2 columns")
            continue

        frame = df.iloc[:, 0].values.astype(float)
        speed = df.iloc[:, 1].values.astype(float)

        if len(frame) != len(speed) or len(frame) < 10:
            print(f"    Skipping {csv_file}: Frame/speed length mismatch or too short")
            continue

        # --- Convert frame → time (s) ---
        time = frame / sampling_rate

        # --- Median filtering ---
        kernel_size = median_kernel_size(time, window_s=1.0)
        speed = median_filter(speed, size=kernel_size, mode='nearest')

        # --- Baseline normalization ---
        baseline_region = speed[time < 180]
        baseline = np.nanmean(baseline_region)
        if np.isnan(baseline) or baseline == 0:
            print(f"    Skipping {csv_file}: Invalid baseline")
            continue
        speed_norm = speed / baseline

        # --- Fit exponential increase in 200–900s window ---
        mask = (time >= 250) & (time <= 600)
        xdata = time[mask]
        ydata = speed_norm[mask]

        # Remove NaNs/Infs
        valid = np.isfinite(ydata)
        xdata, ydata = xdata[valid], ydata[valid]

        if len(xdata) < 5:
            print(f"    Skipping {csv_file}: not enough valid points in fit window")
            continue

        try:
            # --- Bounds ---
            bounds = ([0, 0.1, 0], [2, 500, 1.5])

            # --- Initial guess (clipped to bounds) ---
            B0 = np.clip(1 - ydata[0], bounds[0][0], bounds[1][0])
            tau0 = 10
            D0 = np.clip(ydata[0], bounds[0][2], bounds[1][2])
            p0 = [B0, tau0, D0]

            # --- Fit ---
            popt, _ = curve_fit(exp_increase, xdata, ydata, p0=p0, bounds=bounds)
            B, tau_inc, D = popt
            
            # --- Plot ---
            plt.figure(figsize=(6, 4))
            plt.plot(time, speed_norm, label="Normalized Trace", color='blue')
            plt.plot(xdata, exp_increase(xdata, *popt), label="Fit", color='green', linestyle='--')
            plt.xlabel("Time (s)", fontsize=11)
            plt.ylabel("Normalized Speed", fontsize=11)
            plt.title(f"{subfolder} | {csv_file}\ntau_inc={tau_inc:.3f}", fontsize=11)
            plt.legend(fontsize=11)
            plt.ylim(0, 1.5)
            plt.xticks(np.arange(0, max(time)+1, 200), fontsize=11)
            plt.yticks(fontsize=11)
            plt.tight_layout()
            plt.show()

            print(f"    {csv_file}: tau_inc = {tau_inc:.3f}, B = {B:.2f}, D = {D:.2f}")

            # --- Store results ---
            results.append({
                "Concentration_mM": concentration,
                "File": csv_file,
                "tau_inc": tau_inc,
                "B": B,
                "D": D,
                "Plateau": B + D
            })

        except Exception as e:
            print(f"    Fit failed for {csv_file}: {e}")
            continue

# --- Save results ---
results_df = pd.DataFrame(results)
if results_df.empty:
    print("\n⚠️ No successful fits. No CSV files saved.")
else:
    results_df[["Concentration_mM", "File", "tau_inc"]].to_csv(output_dir / "individual_tau_inc_values.csv", index=False)
    results_df[["Concentration_mM", "File", "B"]].to_csv(output_dir / "individual_B_values.csv", index=False)
    results_df[["Concentration_mM", "File", "D"]].to_csv(output_dir / "individual_D_values.csv", index=False)
    results_df[["Concentration_mM", "File", "Plateau"]].to_csv(output_dir / "individual_plateau_values.csv", index=False)
    print(f"\n✅ Saved individual fit parameter CSV files to: {output_dir.resolve()}")

    # --- Summary per parameter ---
    for param in ["tau_inc", "B", "Plateau"]:
        grouped = results_df.groupby("Concentration_mM")[param]
        summary_df = pd.DataFrame({
            "Concentration_mM": grouped.mean().index,
            f"Mean_{param}": grouped.mean().values,
            f"Std_{param}": grouped.std().values
        })
        summary_df.to_csv(output_dir / f"{param}_summary_per_condition.csv", index=False)

        print(f"\n📊 Average {param} per Condition:")
        for conc, values in grouped:
            print(f"  {conc} mM: {param} = {values.mean():.3f} ± {values.std():.3f}")

### Archived upstream-dependent cell

This original code cell depended on local upstream files or intermediate products that are intentionally not included in this time-series-first supplement. The supported workflow starts from `data/time-series/` and writes regenerated products under `outputs/`.
